# Referencia- és kiterjesztett implementáció konzisztencia-ellenőrzése

Ez a notebook a **„A Crouch-féle szimulációs prevalencia-becslés kiterjesztése folytonos időbeli modellezésre”** című szakdolgozat validációs anyaga.  
A szakdolgozat a Crouch-féle incidencia–túlélés alapú Monte Carlo prevalenciabecslés elméleti keretére és a meglévő `rprev` referencia-implementációra épül. A központi feladat a referencia-eljárás több indexdátum együttes kezelésére szolgáló kiterjesztése, valamint ennek módszertani és numerikus validálása (lásd: 3–5. fejezet).

Ennek megfelelően a **referencia** implementáció definíció szerint egyetlen indexdátumhoz ad prevalencia-becslést. A több időpontra vonatkozó becslés problémáját önálló módszertani kérdésként nem tárgyalja, így több indexdátum esetén egymástól független, ismételt egyindexű futtatásokkal adható eredmény. A **kiterjesztett implementáció** ezzel szemben explicit módon több indexdátum együttes kezelésére készült, és a teljes indexrácsra konzisztens becslést ad egy közös futási keretben (lásd: 4. fejezet – referencia implementáció és a kiterjesztés tárgya; 5. fejezet – matematikai modell és csomagszintű illesztés).

## Validációs célok

A notebook célja annak vizsgálata, hogy a referencia implementáció és a több indexdátumot kezelő kiterjesztett implementáció azonos bemenetek (adat, modellek), azonos indexdátumok és azonos fő paraméterek mellett összevethető, és a megfigyelt eltérések módszertanilag értelmezhetők-e.

A jelölésekben `K` az indexdátumok számát jelöli: `K=1` egyetlen indexdátumot, `K>1` több indexdátumot jelent.

1. **K=1 összevetés (egyindexű viszonyítás)**  
Egyetlen indexdátum mellett mindkét implementációt ugyanarra a horizontkészletre futtatjuk, és a horizontonként kapott abszolút prevalencia-becsléseket vetjük össze.  
A cél annak ellenőrzése, hogy a különbségek nagyságrendje és iránya összhangban áll-e a kiterjesztés ismert szerkezeti eltéréseivel (különösen az esetszintű `xi` küszöb-alapú státuszképzéssel és a véletlenszám-felhasználás eltérő szervezésével), és nem utal-e szisztematikus torzításra (lásd: 5. fejezet – numerikus operacionalizálás; 6. fejezet – validációs eredmények).

2. **K>1 időbeli koherencia (több indexdátum együttes kezelése)**  
Több indexdátum mellett a kiterjesztett implementáció egy közös futási keretben ad becslést az indexrács minden pontjára, míg a referencia megoldás csak indexdátumonként ismételt, egymástól független futtatásokkal képez viszonyítási sorozatot.  
A vizsgálat fókusza annak értékelése, hogy az indexdátumok mentén megjelenő eltérések mintázata összhangban áll-e a kiterjesztés definiált működésével, vagyis azzal, hogy a státuszképzés futáson belül időben konzisztens módon, közös szimulált incidens populációra történik (lásd: 5. fejezet; 6. fejezet).

## Módszertani eltérések és értelmezés

A két implementáció futási szerkezete eltér, ezért a véletlenszám-generátor hívásainak száma és ütemezése sem azonos (lásd dolgozat 5. fejezet). Ennek következtében azonos seed (véletlengenerátor magja) beállítás mellett sem várható bitazonos kimenet.  
A megfigyelt különbségek akkor tekinthetők elfogadhatónak, ha:
- nagyságuk a becslési bizonytalansággal összemérhető,
- nem mutatnak tartós, egyirányú torzítást az indexdátumok mentén,
- és összhangban állnak a kiterjesztés ismert szerkezeti sajátosságaival (lásd: 6–7. fejezet).
- nem haladják meg azt a változékonysági mértéket, amelyet önmagában a seed megválasztása okoz az eljárás kimenetében.


## Notebook felépítése

1. Közös paraméterezés rögzítése.
2. A referencia implementáció futtatása K=1 és K>1 esetekre indexdátumonként, különálló hívásokkal.
3. A kiterjesztett implementáció futtatása K=1 és K>1 esetekre.
4. A K=1 és K>1 eseek célzott öszevetése és kiértékelése.
5. Eltérések esetén kiegészítő diagnosztikák futtatása az eltérések forrásának azonosítására.


---
#### 1. Közös paraméter konfiguráció
- A referencia és a kiterjesztett megoldása futtatása ugyanebből a paraméterkészletből dolgozik, így az összevetés konzisztens.

In [1]:
# Common configuration (shared across runs)
seeds <- c(20260302, 20260303, 20260304)
seed <- seeds[1]
N_boot <- 1000L
num_years <- c(3, 5, 10, 13, 15, 20, 25)

# Index date grid (K = number of index dates)
K <- 3L
index_start <- as.Date("2012-01-07")
index_dates <- seq.Date(from = index_start, by = "year", length.out = K)

# Model + prevalence settings
inc_formula <- entrydate ~ sex
surv_formula <- survival::Surv(time, status) ~ age + sex
dist <- "weibull"
population_size <- 1e6
death_column <- "eventdate"
status_col <- "status"


#### 2. Környezet előkészítése
- Betöltjük a szükséges csomagokat (`rprev`, `survival`, `lubridate`), valamint a `prevsim` példaadatot, hogy a referenciafuttatások egységes környezetben és bemeneten induljanak.
- A betöltött adatokból előállítjuk a futtatásokhoz használt munkatáblát (`base_df`), amelyet a későbbi összevetések során konzisztensen használunk.

In [2]:
# Csomagok betöltése
suppressPackageStartupMessages({
  library(rprev)
  library(survival)
  library(lubridate)
})

# Példaadat betöltése (kiinduló tábla)
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)


Warning message:
"package 'rprev' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"


#### 3. Referencia `rprev` futtatás (K=1 alap eset, K>1 ismételt futások)
- A referencia implementáció `prevalence()` függvénye egyetlen indexdátumra ad becslést. Több indexdátum esetén a viszonyítási eredmény indexdátumonként ismételt különálló futtatásokból áll elő.
- A K>1 viszonyítási eredmények és a kiterjesztett eljárással való teljes összevetés automatizálása ebben a blokkban még nem végleges, itt kizárólag a futtatások előkészítése és a bemenetek rögzítése történik.
- A kiértékelést több seed beállítás mellett is elvégezzük.


In [3]:
# K=1 (egy indexdátum)
index_k1 <- index_dates[1]
registry_start_ref <- min(prevsim$entrydate, na.rm = TRUE)

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  set.seed(s)
  res_ref_k1 <- rprev::prevalence(
    index = index_k1,
    num_years_to_estimate = num_years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_ref,
    N_boot = N_boot
  )

  print(summary(res_ref_k1))
}




--------------------------------------------------------------------------------
seed = 20260302 
Prevalence 
~~~~~~~~~~
Estimated prevalence at 2012-01-07:
3 years: 207 (20.7 per 1e+05) 
5 years: 297 (29.7 per 1e+05) 
10 years: 508.49 (50.85 per 1e+05) 
13 years: 598.4 (59.84 per 1e+05) 
15 years: 650.73 (65.07 per 1e+05) 
20 years: 762.22 (76.22 per 1e+05) 
25 years: 850.62 (85.06 per 1e+05) 

Registry Data
~~~~~~~~~~~~~
Index date: 2012-01-07 
Start date: 2003-01-07 
Overall incidence rate: 0.304 
Counted prevalent cases: 475 

Simulation
~~~~~~~~~~
Iterations: 1000 
Average incidence rate: 0.273 
P-value: 0.6004299NULL

--------------------------------------------------------------------------------
seed = 20260303 
Prevalence 
~~~~~~~~~~
Estimated prevalence at 2012-01-07:
3 years: 207 (20.7 per 1e+05) 
5 years: 297 (29.7 per 1e+05) 
10 years: 508.26 (50.83 per 1e+05) 
13 years: 597.34 (59.73 per 1e+05) 
15 years: 650.02 (65 per 1e+05) 
20 years: 762.02 (76.2 per 1e+05) 
25 years

In [ ]:
# K>1 (több indexdátum, ismételt K=1 futások)
registry_start_ref <- min(prevsim$entrydate, na.rm = TRUE)

# Gyors áttekintés: abszolút prevalencia indexdátumonként, horizontonként
extract_abs <- function(res) {
  out <- sapply(num_years, function(y) {
    as.numeric(res$estimates[[paste0("y", y)]][["absolute.prevalence"]])
  })
  names(out) <- paste0("y", num_years)
  out
}

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  res_ref_by_index_s <- lapply(index_dates, function(idx) {
    set.seed(s)
    rprev::prevalence(
      index = idx,
      num_years_to_estimate = num_years,
      data = base_df,
      inc_formula = inc_formula,
      surv_formula = surv_formula,
      dist = dist,
      population_size = population_size,
      death_column = death_column,
      registry_start_date = registry_start_ref,
      N_boot = N_boot
    )
  })
  names(res_ref_by_index_s) <- as.character(index_dates)

  ref_abs_table_s <- t(sapply(res_ref_by_index_s, extract_abs))
  print(ref_abs_table_s)

  if (s == seeds[1]) {
    res_ref_by_index <- res_ref_by_index_s
    ref_abs_table <- ref_abs_table_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményeit használjuk
# ref_abs_table



--------------------------------------------------------------------------------
seed = 20260302 
            y3  y5    y10   y13    y15    y20    y25
2012-01-07 207 297 508.49 598.4 650.73 762.22 850.62
2013-01-07 207 304 517.00 606.0 658.32 769.98 858.73
2014-01-07 127 226 433.00 530.3 582.69 694.12 782.52

--------------------------------------------------------------------------------
seed = 20260303 
            y3  y5    y10    y13    y15    y20    y25
2012-01-07 207 297 508.26 597.34 650.02 762.02 851.34
2013-01-07 207 304 517.00 606.35 658.66 770.77 860.07
2014-01-07 127 226 433.00 529.76 582.51 694.44 783.76

--------------------------------------------------------------------------------
seed = 20260304 
            y3  y5   y10    y13    y15    y20    y25
2012-01-07 207 297 508.1 597.32 650.09 761.20 849.95
2013-01-07 207 304 517.0 606.36 658.96 770.27 859.16
2014-01-07 127 226 433.0 529.83 582.68 693.70 782.45


,y3,y5,y10,y13,y15,y20,y25
2012-01-07,207,297,508.49,598.4,650.73,762.22,850.62
2013-01-07,207,304,517.00,606.0,658.32,769.98,858.73
2014-01-07,127,226,433.00,530.3,582.69,694.12,782.52


#### 4. Kiterjesztett rprev csomag betöltése lokális forrásból és git-meta rögzítése
- A csomag lokális forrásból kerül betöltésre, így a futtatás ténylegesen a vizsgált, módosított implementációval történik.
- A futtatott verzió azonosíthatóságát a git ág és commit azonosító kiírása biztosítja.

In [5]:
# Projekt lokális gyökérkönyvtárának meghatározása (git repo root)
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# Telepített rprev leválasztása, majd lokális forrásból betöltés
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")

suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))

# Betöltött csomag elérési útjának megjelenítése
cat(
  "Loaded rprev from: ",
  # normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"),
  # "\n",
  sep = ""
)

Warning message:
"package 'testthat' was built under R version 4.4.3"


Loaded rprev from: 

In [6]:
# Git meta rögzítése: futtatott kódverzió azonosítása
git_cmd <- function(args) {
  out <- suppressWarnings(
    tryCatch(system2("git", args, stdout = TRUE, stderr = TRUE), error = function(e) character(0))
  )
  status <- attr(out, "status")
  if (!is.null(status) || length(out) == 0) return(NA_character_)
  trimws(out[[1]])
}

old_wd <- getwd()                         # munkakönyvtár mentése
on.exit(setwd(old_wd), add = TRUE)        # visszaállítás a cella végén
setwd(project_root)                       # git parancsok futtatása a repo gyökeréből

branch <- git_cmd(c("rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")

Git branch: notebooks/ref-ext-consistency
Git commit: 099fb43d177ecb881bc976e63287f1222c9fdab1


#### 5. Kiterjesztett `rprev` futtatás (K=1 és natív többindexű becslés)
- Az előző blokkban betöltött lokális implementáció `prevalence()` függvényét lefuttatjuk ugyanazzal az indexdátum- és paraméterkészlettel, mint a referenciafuttatásoknál.
- K=1 esetben egyetlen indexdátumra, K>1 esetben pedig natív többindexű hívással futtatunk, majd egy rövid táblában összefoglaljuk az abszolút prevalenciákat indexdátumonként és horizontonként.
- A kiértékelést több seed beállítás mellett is elvégezzük.

In [ ]:
# K=1 (egy indexdátum) - lokális, kiterjesztett implementáció
index_k1 <- index_dates[1]
registry_start_ext <- min(base_df$entrydate, na.rm = TRUE)

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  set.seed(s)
  res_ext_k1_s <- prevalence(
    index = index_k1,
    num_years_to_estimate = num_years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_ext,
    N_boot = N_boot
  )
  print(summary(res_ext_k1_s))

  if (s == seeds[1]) {
    res_ext_k1 <- res_ext_k1_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményét használjuk
summary(res_ext_k1)




--------------------------------------------------------------------------------
seed = 20260302 
Prevalence 
~~~~~~~~~~
Estimated prevalence:
 index_date years absolute.prevalence per100K per100K.upper per100K.lower
 2012-01-07     3              207.00   20.70         23.52         17.88
 2012-01-07     5              297.00   29.70         33.08         26.32
 2012-01-07    10              508.18   50.82         56.29         45.35
 2012-01-07    13              597.62   59.76         66.63         52.90
 2012-01-07    15              649.94   64.99         72.59         57.40
 2012-01-07    20              761.37   76.14         85.48         66.80
 2012-01-07    25              850.17   85.02         95.83         74.21

Registry Data
~~~~~~~~~~~~~
Index date(s): 2012-01-07 
Start date: 2003-01-07 
Overall incidence rate: 0.304 
Counted prevalent cases:
 index_date counted
 2012-01-07     475

Simulation
~~~~~~~~~~
Iterations: 1000 
Average incidence rate: 0.273 
P-value: 0.60042

In [ ]:
# K>1 (natív többindexű futtatás) - lokális, kiterjesztett implementáció
# Gyors áttekintés: abszolút prevalencia indexdátumonként, horizontonként
years_lbl <- paste0("y", num_years)

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  set.seed(s)
  res_ext_kmulti_s <- prevalence(
    index = index_dates,
    num_years_to_estimate = num_years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_ext,
    N_boot = N_boot
  )
  print(summary(res_ext_kmulti_s))

  ext_abs_table_s <- do.call(
    cbind,
    lapply(num_years, function(y) {
      est <- res_ext_kmulti_s$estimates[[paste0("y", y)]]
      est <- est[order(est$index_date), , drop = FALSE]
      as.numeric(est[["absolute.prevalence"]])
    })
  )
  colnames(ext_abs_table_s) <- years_lbl
  rownames(ext_abs_table_s) <- as.character(sort(unique(res_ext_kmulti_s$index_dates)))
  print(ext_abs_table_s)

  if (s == seeds[1]) {
    res_ext_kmulti <- res_ext_kmulti_s
    ext_abs_table <- ext_abs_table_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményeit használjuk
# ext_abs_table


--------------------------------------------------------------------------------
seed = 20260302 
Prevalence 
~~~~~~~~~~
Estimated prevalence:
 index_date years absolute.prevalence per100K per100K.upper per100K.lower
 2012-01-07     3              207.00   20.70         23.52         17.88
 2012-01-07     5              297.00   29.70         33.08         26.32
 2012-01-07    10              508.34   50.83         56.28         45.39
 2012-01-07    13              597.94   59.79         66.63         52.96
 2012-01-07    15              650.40   65.04         72.65         57.43
 2012-01-07    20              762.33   76.23         85.48         66.98
 2012-01-07    25              851.55   85.16         95.95         74.36
 2013-01-07     3              207.00   20.70         23.52         17.88
 2013-01-07     5              304.00   30.40         33.82         26.98
 2013-01-07    10              517.00   51.70         56.16         47.24
 2013-01-07    13              606.78   60

,y3,y5,y10,y13,y15,y20,y25
2012-01-07,207,297,508.34,597.94,650.40,762.33,851.55
2013-01-07,207,304,517.00,606.78,659.01,771.23,860.29
2014-01-07,127,226,433.00,530.04,582.81,695.24,784.15


#### Összehasonlító tábla (referencia vs kiterjesztett)
- Az alábbi táblázat az abszolút prevalencia-becsléseket veti össze indexdátumonként és időhorizontonként.
- A `counted` oszlop a regiszterből közvetlenül számlált prevalens esetszámot mutatja (indexdátumonként), a `ref` és `ext` oszlopok pedig az eljárások becsléseit.
- A `delta` és a `delta_pct` oszlopok a referencia-megoldást tekintik bázisnak: `delta = kiterjesztett - referencia`, `delta_pct = 100 * delta / referencia`.


In [9]:
# Dinamikus összevetés: referencia (ismételt K=1) vs kiterjesztett (natív többindexű)
ref_mat <- ref_abs_table
ext_mat <- ext_abs_table

idx <- intersect(rownames(ref_mat), rownames(ext_mat))
yrs <- intersect(colnames(ref_mat), colnames(ext_mat))

compare_df <- expand.grid(index_date = idx, horizon = yrs, stringsAsFactors = FALSE)
compare_df$ref <- mapply(function(r, c) as.numeric(ref_mat[r, c]), compare_df$index_date, compare_df$horizon)
compare_df$ext <- mapply(function(r, c) as.numeric(ext_mat[r, c]), compare_df$index_date, compare_df$horizon)

counted_by_date <- sapply(res_ref_by_index, function(res) as.numeric(res$counted))
counted_by_date <- counted_by_date[as.character(idx)]
compare_df$counted <- as.numeric(counted_by_date[compare_df$index_date])
compare_df$delta <- compare_df$ext - compare_df$ref
compare_df$delta_pct <- ifelse(compare_df$ref == 0, NA_real_, 100 * compare_df$delta / compare_df$ref)

compare_df$horizon_years <- as.integer(sub("^y", "", compare_df$horizon))
compare_df <- compare_df[order(as.Date(compare_df$index_date), compare_df$horizon_years), ]
compare_df$horizon <- NULL
compare_df <- compare_df[, c("index_date", "horizon_years", "counted", "ref", "ext", "delta", "delta_pct")]

# Megjelenítéshez kerekítés (a counted esetszám marad egész)
compare_df$ref <- round(compare_df$ref, 2)
compare_df$ext <- round(compare_df$ext, 2)
compare_df$delta <- round(compare_df$delta, 2)
compare_df$delta_pct <- round(compare_df$delta_pct, 2)

# Sorindex elrejtése: data.table megjelenítés rownames nélkül
compare_dt <- data.table::as.data.table(compare_df)
compare_dt


index_date,horizon_years,counted,ref,ext,delta,delta_pct
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2012-01-07,3,475,207.00,207.00,0.00,0.00
2012-01-07,5,475,297.00,297.00,0.00,0.00
2012-01-07,10,475,508.49,508.34,-0.15,-0.03
2012-01-07,13,475,598.40,597.94,-0.46,-0.08
2012-01-07,15,475,650.73,650.40,-0.33,-0.05
2012-01-07,20,475,762.22,762.33,0.11,0.01
2012-01-07,25,475,850.62,851.55,0.93,0.11
2013-01-07,3,517,207.00,207.00,0.00,0.00
2013-01-07,5,517,304.00,304.00,0.00,0.00


## 4) Prefix-hatas vizsgalata (horizont-bovites)
- Ugyanarra az indexdatum-racsra lefuttatjuk a kiterjesztett eljarast novekvo horizonkészletekkel, amelyeket a `num_years` vektor prefixei adnak.
- Az eredmeny egy matrix: az oszlopok a prefix-horizonkészletekhez tartozo futasok, a sorok a `num_years` altal megadott horizontok. A kozos horizontok ertekeinek stabilitasa a prefix-invariancia indikatora.


In [10]:
# Horizont-bovites: a num_years prefixeit futtatjuk vegig
index_focus <- index_dates[1]
registry_start_local <- min(base_df$entrydate, na.rm = TRUE)

horizon_sets <- lapply(seq_along(num_years), function(i) num_years[seq_len(i)])
names(horizon_sets) <- vapply(horizon_sets, function(v) paste(v, collapse = "_"), character(1))

run_once <- function(years) {
  set.seed(seed)
  prevalence(
    index = index_dates,
    num_years_to_estimate = years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_local,
    N_boot = N_boot
  )
}

extract_abs <- function(res, years) {
  out <- sapply(years, function(y) {
    est <- res$estimates[[paste0("y", y)]]
    as.numeric(est[est$index_date == index_focus, "absolute.prevalence"][1])
  })
  names(out) <- as.character(years)
  out
}

runs <- lapply(horizon_sets, run_once)
vals <- mapply(function(res, years) extract_abs(res, years), runs, horizon_sets, SIMPLIFY = FALSE)

all_years <- sort(unique(num_years))
prefix_mat <- matrix(
  NA_real_,
  nrow = length(all_years),
  ncol = length(vals),
  dimnames = list(as.character(all_years), names(vals))
)
for (nm in names(vals)) {
  prefix_mat[names(vals[[nm]]), nm] <- round(vals[[nm]], 2)
}

prefix_mat


,3,3_5,3_5_10,3_5_10_13,3_5_10_13_15,3_5_10_13_15_20,3_5_10_13_15_20_25
3,207,207,207.00,207.00,207.00,207.00,207.00
5,NA,297,297.00,297.00,297.00,297.00,297.00
10,NA,NA,508.28,508.52,508.48,508.02,508.34
13,NA,NA,NA,598.25,598.13,598.46,597.94
15,NA,NA,NA,NA,650.86,650.68,650.40
20,NA,NA,NA,NA,NA,761.97,762.33
25,NA,NA,NA,NA,NA,NA,851.55


#### 5) Seed-variabilitás vizsgálata (fix horizont, ismételt futások)

Ebben a blokkban azt becsüljük meg, hogy az abszolút prevalencia-becslés mekkora ingadozást mutat pusztán a véletlenszám-generátor magjának (seed) megváltoztatása miatt, miközben minden más beállítás változatlan.

- Rögzítünk egy vizsgált időhorizontot (pl. 13 év), és ugyanarra az indexdátum-rácsra többször lefuttatjuk a kiterjesztett `prevalence()` hívást eltérő seedekkel.
- Az összefoglaló táblában az oszlopok az egyes seed-futtatások abszolút prevalencia-becsléseit tartalmazzák, az utolsó oszlop pedig indexdátumonként a futások közötti varianciát adja.

A kapott variancia közvetlen viszonyítási alapot ad ahhoz, hogy a referencia és a kiterjesztett megoldás közötti eltérések nagyságrendje mennyiben tekinthető a seed-választásból adódó szokásos változékonyságnak megfelelőnek.


In [11]:
# Variabilitás: fix horizon, több seed futtatás

# Paraméterek a seed-szenzitivitás vizsgálathoz
h_focus <- 13
n_seed_runs <- 8
seed_base <- seed
seeds_var <- seed_base + seq_len(n_seed_runs) - 1

registry_start_local <- min(base_df$entrydate, na.rm = TRUE)

run_one_seed <- function(seed_i) {
  set.seed(seed_i)
  res <- prevalence(
    index = index_dates,
    num_years_to_estimate = c(h_focus),
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_local,
    N_boot = N_boot
  )
  est <- res$estimates[[paste0("y", h_focus)]]
  data.frame(
    index_date = as.Date(est$index_date),
    abs_prev = as.numeric(est[["absolute.prevalence"]])
  )
}

runs <- lapply(seeds_var, run_one_seed)

# Széles tábla: 1 oszlop / seed, soronként indexdátum
seed_tbl <- data.frame(index_date = as.Date(index_dates))
for (i in seq_along(seeds_var)) {
  df_i <- runs[[i]]
  v_i <- df_i$abs_prev[match(as.Date(index_dates), as.Date(df_i$index_date))]
  seed_tbl[[paste0("seed_", seeds_var[i])]] <- v_i
}
seed_tbl$horizon_years <- h_focus

# Variancia az ismételt seed-futtatások között (indexdátumonként)
run_cols <- grep("^seed_", names(seed_tbl), value = TRUE)
seed_tbl$variance <- apply(seed_tbl[, run_cols, drop = FALSE], 1, var, na.rm = TRUE)

# Megjelenítés (kerekítve)
seed_tbl[, run_cols] <- lapply(seed_tbl[, run_cols, drop = FALSE], function(x) round(x, 2))
seed_tbl$variance <- round(seed_tbl$variance, 2)

# Oszlopsorrend: indexdátum, horizont, seed futások, variancia
seed_tbl <- seed_tbl[, c("index_date", "horizon_years", run_cols, "variance")]
data.table::as.data.table(seed_tbl)


index_date,horizon_years,seed_20260302,seed_20260303,seed_20260304,seed_20260305,seed_20260306,seed_20260307,seed_20260308,seed_20260309,variance
<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2012-01-07,13,598.25,597.55,598.58,598.20,597.71,598.14,598.67,597.49,0.20
2013-01-07,13,606.96,606.30,606.94,606.87,606.47,606.62,606.95,606.16,0.10
2014-01-07,13,530.41,529.97,530.28,530.32,529.87,530.04,530.52,529.87,0.06


#### Megállapítás

A fenti prefix-tábla oszlopaiban ugyanazt az eljárást futtatjuk eltérő horizontkészletekkel (a `num_years` vektor prefixeivel), miközben minden futtatás előtt ugyanazt a véletlenszám-generátor magot (seedet) állítjuk be. Ennek ellenére a közös horizontokra kapott becslések nem feltétlenül egyeznek, mert a horizontkészlet megváltoztatása a szimuláció teljes lefutását is megváltoztatja.

A fő ok a véletlenszám-felhasználás eltérő ütemezése és mennyisége:

- A `prevalence()` a szimuláció induló dátumát a **legnagyobb kért horizont** alapján állítja be: `sim_start_date <- min(index_dates) - years(max(num_years_to_estimate))` (R/prevalence.R). Ha a horizontlistát bővítjük, a `sim_start_date` korábbra kerül.
- A `sim_prevalence()` ezután a szimulációs ablak hosszát a `starting_date` és a legkésőbbi indexdátum különbségéből számolja: `number_incident_days <- difftime(max(index_dates), starting_date, ...)` (R/prevalence.R). A hosszabb ablak **más (és tipikusan több) szimulált incidens esetet** eredményezhet a `draw_incident_population(...)` hívásban, ami a véletlenszám-generátor állapotát is másképp "fogyasztja".
- A kiterjesztett Monte Carlo magban futásonként egyszer sorsolunk esetszintű küszöböt: `xi <- runif(nrow(incident_population))`, majd ezt az összes indexdátumhoz felhasználjuk: `alive_k <- I(xi <= p_k)` (R/prevalence.R). Ha a szimulált incidens populáció mérete (és így a `xi` hossza) megváltozik, akkor ugyanazon kezdő seed mellett is eltérő RNG-állapotból folytatódnak a további lépések.

A referencia (egyindexű) megoldáshoz képest a kiterjesztés a státuszképzés véletlen komponensét is másképp szervezi: a túlélési valószínűségekhez tartozó bináris státusz nem független Bernoulli-sorsolások sorozataként áll elő, hanem az esetszintű `xi ~ Unif(0,1)` küszöb és a `p_k` túlélési valószínűségek összehasonlításával. Ez K=1 esetben is megváltoztatja a véletlenszám-hívások mintázatát, ezért ugyanazon seed mellett sem várható szigorú numerikus egyezés.

Módszertani szempontból a kiterjesztés célja az, hogy **egy futáson belül** az indexdátumok mentén időben konzisztens státuszképzés történjen (dolgozat 5. fejezet: esetszintű `xi` küszöb és a Delta-mátrix/maszk konstrukció). Ez a tulajdonság nem garantálja, hogy **külön futtatások** (eltérő horizontkészletek) között a közös horizontokra kapott becslések változatlanok maradnak.

Gyakorlati értelmezés: a prefix-oszlopok közötti különbség lényegében azt jelenti, hogy a közös horizont becslése egy másik (ugyanazzal a maggal indított, de másképp felhasznált) véletlenszám-sorozaton alapul. Emiatt a különbségek nagyságrendje tipikusan összemérhető azzal a variabilitással, amit különböző seed-beállítások is okoznának.


#### Tovabbi prefix-vizsgalat (3 horizon: kozepso stabilitasa)
- A `num_years` vektorbol kivalasztunk harom szomszedos idohorizontot (kisebb, kozepso, nagyobb), es azt vizsgaljuk, hogy a kozepso horizonthoz tartozo abszolut prevalencia-becslest mikent befolyasolja a horizonkeszlet bovitesi modja.
- Negy konfiguraciot futtatunk: 
    - (i) csak a kozepso horizont, 
    - (ii) a kisebb+kozepso, 
    - (iii) a kozepso+nagyobb, 
    - (iv) a kisebb+kozepso+nagyobb egyuttesen. 
- Az osszefoglalo tabla indexdatumonkent a kozepso horizon becsleset, valamint a „csak kozepso” konfiguraciohoz viszonyitott elterest (delta, delta%) kozli.




In [12]:
# tovabbi: 3 horizonon a kozepso stabilitasa (egyedul / ele / moge / ele+mogé)
h3 <- num_years[2:4]
h_small <- h3[1]
h_mid <- h3[2]
h_large <- h3[3]

year_sets <- list(
  `kozepso` = c(h_mid),
  `kisebb_kozepso` = c(h_small, h_mid),
  `kozepso_nagyobb` = c(h_mid, h_large),
  `kisebb_kozepso_nagyobb` = c(h_small, h_mid, h_large)
)

registry_start_local <- min(base_df$entrydate, na.rm = TRUE)

run_ext <- function(years) {
  set.seed(seed)
  prevalence(
    index = index_dates,
    num_years_to_estimate = years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_local,
    N_boot = N_boot
  )
}

extract_mid_abs <- function(res) {
  est <- res$estimates[[paste0("y", h_mid)]]
  data.frame(
    index_date = as.Date(est$index_date),
    abs_prev_mid = as.numeric(est[["absolute.prevalence"]])
  )
}

res_list <- lapply(year_sets, run_ext)
mid_list <- lapply(res_list, extract_mid_abs)

summary_df <- do.call(
  rbind,
  Map(function(nm, yrs, df) {
    df$case <- nm
    df$years_set <- paste(yrs, collapse = ",")
    df$h_mid <- h_mid
    df
  }, names(year_sets), year_sets, mid_list)
)

base_mid <- summary_df[summary_df$case == "kozepso", c("index_date", "abs_prev_mid")]
names(base_mid)[2] <- "abs_prev_mid_base"
summary_df <- merge(summary_df, base_mid, by = "index_date", all.x = TRUE)
summary_df$delta <- summary_df$abs_prev_mid - summary_df$abs_prev_mid_base
summary_df$delta_pct <- ifelse(summary_df$abs_prev_mid_base == 0, NA_real_, 100 * summary_df$delta / summary_df$abs_prev_mid_base)

case_order <- names(year_sets)
summary_df$case <- factor(summary_df$case, levels = case_order)
summary_df <- summary_df[order(summary_df$index_date, summary_df$case), ]

summary_df$abs_prev_mid <- round(summary_df$abs_prev_mid, 2)
summary_df$abs_prev_mid_base <- round(summary_df$abs_prev_mid_base, 2)
summary_df$delta <- round(summary_df$delta, 2)
summary_df$delta_pct <- round(summary_df$delta_pct, 2)

summary_df[, c("index_date", "h_mid", "case", "years_set", "abs_prev_mid_base", "abs_prev_mid", "delta", "delta_pct")]


,index_date,h_mid,case,years_set,abs_prev_mid_base,abs_prev_mid,delta,delta_pct
,<date>,<dbl>,<fct>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,2012-01-07,10,kozepso,10,508.28,508.28,0.00,0.00
4,2012-01-07,10,kisebb_kozepso,"5,10",508.28,508.28,0.00,0.00
3,2012-01-07,10,kozepso_nagyobb,"10,13",508.28,508.52,0.24,0.05
2,2012-01-07,10,kisebb_kozepso_nagyobb,"5,10,13",508.28,508.52,0.24,0.05
6,2013-01-07,10,kozepso,10,517.00,517.00,0.00,0.00
5,2013-01-07,10,kisebb_kozepso,"5,10",517.00,517.00,0.00,0.00
8,2013-01-07,10,kozepso_nagyobb,"10,13",517.00,517.00,0.00,0.00
7,2013-01-07,10,kisebb_kozepso_nagyobb,"5,10,13",517.00,517.00,0.00,0.00
11,2014-01-07,10,kozepso,10,433.00,433.00,0.00,0.00
